<!-- SPDX-License-Identifier: CC-BY-4.0 -->
<!-- Copyright Contributors to the ODPi Egeria project 2024. -->

![Egeria Logo](https://raw.githubusercontent.com/odpi/egeria/main/assets/img/ODPi_Egeria_Logo_color.png)

### Coco Pharmaceuticals Lab
----

# Adding the Procurement business capability

## Introduction

As part of the sustainability initiative, [Tom Tally](https://egeria-project.org/practices/coco-pharmaceuticals/personas/tom-tally/) notices that the Procurement function is not represented in the sustainability community.  Purchasing decisions shape much of the company's environmental footprint, so this is a gap.

Procurement has so far been a small team within Finance, managed by [Sally Counter](https://egeria-project.org/practices/coco-pharmaceuticals/personas/sally-counter/).  That is changing.  The supplier fraud showed how exposed Coco Pharmaceuticals is when its supplier records are not authoritative: the fraud travelled through the gap between the many places a supplier could be created and paid.  Coco core alone buys through several purchasing systems, and the two acquisitions each arrived with their own supplier lists - Austin in SAP Ariba and Informatica MDM, Bucharest in SAP S/4HANA.  So Procurement is being expanded into a business area of its own, responsible for managing the company's suppliers.

This notebook:

1. Adds a new **Procurement** business capability to the *Coco Pharmaceuticals Business Areas*.  Its only member is the existing **Procurement Team**, led by Sally.
2. Creates a **Procurement Sustainability Champion** role by copying the *Finance Sustainability Champion* role, and appoints Sally to it.
3. Adds the people responsible for procurement at the Austin and Bucharest sites to the Procurement Team.  Together with Sally, they will consolidate the supplier lists.

It needs [adding-acquisition-employees.ipynb](adding-acquisition-employees.ipynb) to have been run first.

----

## Setting up

The first code cell sets up the connection to Egeria.  If you are using the standard quickstart environment then there is no need to change these values.

Tom uses the [Digital Business](https://egeria-project.org/services/omvs/digital-business/overview/) client to create the business capability, the [Collection Manager](https://egeria-project.org/services/omvs/collection-manager/overview/) client to add it to the business areas, and the [Actor Manager](https://egeria-project.org/services/omvs/actor-manager/overview/) client to create and fill the champion role.  [Faith Broker](https://egeria-project.org/practices/coco-pharmaceuticals/personas/faith-broker/) from Human Resources uses the Actor Manager client to add the new members to the Procurement Team.

----

In [ ]:
# Initialize pyegeria

import os
view_server = os.environ.get("EGERIA_VIEW_SERVER","qs-view-server")
url = os.environ.get("EGERIA_VIEW_SERVER_URL","https://host.docker.internal:9443")
user_pwd = os.environ.get("EGERIA_USER_PASSWORD")

toms_user_id = "tomtally"
faiths_user_id = "faithbroker"

print("\n")
print("Accessing view server " + view_server + " on platform " + url + " for users " + toms_user_id + " and " + faiths_user_id)
print("\n")

from IPython.display import display, Markdown
import json

from pyegeria import DigitalBusiness, CollectionManager, ActorManager, ClassificationExplorer
from pyegeria.core._globals import NO_ELEMENTS_FOUND

toms_business_client = DigitalBusiness(view_server, url, toms_user_id, user_pwd)
toms_business_client.create_egeria_bearer_token()

toms_collection_client = CollectionManager(view_server, url, toms_user_id, user_pwd)
toms_collection_client.create_egeria_bearer_token()

toms_actor_client = ActorManager(view_server, url, toms_user_id, user_pwd)
toms_actor_client.create_egeria_bearer_token()

toms_explorer_client = ClassificationExplorer(view_server, url, toms_user_id, user_pwd)
toms_explorer_client.create_egeria_bearer_token()

faiths_actor_client = ActorManager(view_server, url, faiths_user_id, user_pwd)
faiths_actor_client.create_egeria_bearer_token()

print("Connected to Egeria")

----

As in the other notebooks, everything is located through its unique name so that the notebook can be rerun.

----

In [ ]:
def get_guid(qualified_name: str) -> str | None:
    guid = toms_explorer_client.get_element_guid_by_unique_name(qualified_name)
    if guid is None or guid == NO_ELEMENTS_FOUND:
        return None
    return guid


def related_qualified_names(element: dict, relationship_list_name: str) -> list[str]:
    return [related["relatedElement"]["properties"].get("qualifiedName")
            for related in (element.get(relationship_list_name) or [])]


business_areas_qualified_name = "RootCollection::Coco Pharmaceuticals Business Areas"
procurement_team_qualified_name = "Department::6866"
sally_qualified_name = "Person::UK::457911"

----

## Step 1: Adding the Procurement business capability

Each business area of Coco Pharmaceuticals is a *BusinessCapability* of type `BUSINESS_AREA`, and a member of the *Coco Pharmaceuticals Business Areas* collection.  The teams that make up the business area are members of the business capability, with the membership type *Owner*.

----

In [ ]:
identifier = "PROCUREMENT"
displayName = "Procurement"
description = "Purchasing of goods and services for Coco Pharmaceuticals. Responsible for managing supplier relationships."

procurement_qualified_name = "BusinessArea::" + identifier
procurement_guid = get_guid(procurement_qualified_name)

if procurement_guid:
    print(f"{procurement_qualified_name} already exists")
else:
    body = {
        "class": "NewElementRequestBody",
        "isOwnAnchor": True,
        "properties": {
            "class": "BusinessCapabilityProperties",
            "typeName": "BusinessCapability",
            "qualifiedName": procurement_qualified_name,
            "identifier": identifier,
            "displayName": displayName,
            "description": description,
            "businessCapabilityType": "BUSINESS_AREA",
        },
    }
    procurement_guid = toms_business_client.create_business_capability(body)
    toms_collection_client.add_to_collection(get_guid(business_areas_qualified_name), procurement_guid)

    membership_body = {
        "class": "NewRelationshipRequestBody",
        "properties": {
            "class": "CollectionMembershipProperties",
            "membershipType": "Owner",
            "membershipRationale": "Within organization",
        },
    }
    toms_collection_client.add_to_collection(procurement_guid, get_guid(procurement_team_qualified_name), membership_body)
    print(f"{procurement_qualified_name} created and added to {business_areas_qualified_name}, with the Procurement Team as its member")

----

The Procurement Team already exists.  It was created by the Coco Pharmaceuticals archive as a department within the Finance Team, with Sally as its leader.  The team stays within Finance for line management, but it now belongs to the Procurement business area.

----

In [ ]:
procurement_team = toms_actor_client.get_actor_profile_by_guid(get_guid(procurement_team_qualified_name), output_format="JSON")
print(procurement_team["properties"]["displayName"] + ": " + procurement_team["properties"]["description"])

team_leader_role = toms_actor_client.get_actor_role_by_guid(get_guid("TeamLeader::" + procurement_team_qualified_name), output_format="JSON")
for performer in team_leader_role.get("rolePerformers") or []:
    print("Team leader: " + performer["relatedElement"]["properties"]["displayName"])

----

## Step 2: Creating the Procurement Sustainability Champion

Each business area has a sustainability champion.  The champions are *GovernanceRole* elements in the sustainability governance domain (domain 9), and each one is:

* scoped to its business area through the *AssignmentScope* relationship.
* a contributor to the Sustainability Community, also through *AssignmentScope*.
* governed by the *Sustainability Champion* governance responsibility, through the *GovernedBy* relationship.

The quickest way to create the new role is to copy the *Finance Sustainability Champion*.  Any element can be used as a template.  Egeria creates a new element with the template's properties, overlaid with the replacement properties supplied in the request.  It also copies the template's relationships: the new element is linked to the same elements as the template, except for elements anchored to the template, which are copied too.

----

In [ ]:
finance_champion_qualified_name = "governance-representative::FIN-SUSTAINABILITY-CHAMP"
champion_identifier = "PROC-SUSTAINABILITY-CHAMP"
champion_qualified_name = "governance-representative::" + champion_identifier


def string_value(value: str) -> dict:
    return {"class": "PrimitiveTypePropertyValue", "typeName": "string",
            "primitiveTypeCategory": "OM_PRIMITIVE_TYPE_STRING", "primitiveValue": value}


champion_guid = get_guid(champion_qualified_name)
if champion_guid:
    print(f"{champion_qualified_name} already exists")
else:
    body = {
        "class": "TemplateRequestBody",
        "isOwnAnchor": True,
        "templateGUID": get_guid(finance_champion_qualified_name),
        "replacementProperties": {
            "class": "EntityElementProperties",
            "propertyValueMap": {
                "qualifiedName": string_value(champion_qualified_name),
                "identifier": string_value(champion_identifier),
                "displayName": string_value("Procurement Sustainability Champion"),
            },
        },
    }
    champion_guid = toms_actor_client.create_actor_role_from_template(body)
    print(f"{champion_qualified_name} created from {finance_champion_qualified_name}")

----

So the new role arrives linked to the Sustainability Community, the Sustainability Champion responsibility and Sally, who is also the Finance champion - and scoped to Finance.  The code below replaces the Finance scope with Procurement.  It also checks for each of the other relationships, so that the role is complete whatever the template brought with it.

----

In [ ]:
community_qualified_name = "Community::Egeria:ValidMetadataValue::domainIdentifier-(9)"
responsibility_qualified_name = "GovernanceResponsibility::Sustainability Champion"

champion = toms_actor_client.get_actor_role_by_guid(champion_guid, output_format="JSON")
scopes = related_qualified_names(champion, "assignmentScope")
print("Copied from the template: " + ", ".join(scopes + related_qualified_names(champion, "governedBy") + related_qualified_names(champion, "rolePerformers")) or "no relationships")

if "BusinessArea::FIN" in scopes:
    toms_actor_client.detach_assignment_scope(get_guid("BusinessArea::FIN"), champion_guid)
    print("Removed the scope to BusinessArea::FIN")

if procurement_qualified_name not in scopes:
    toms_actor_client.link_assignment_scope(procurement_guid, champion_guid)
    print("Scoped the role to " + procurement_qualified_name)

if community_qualified_name not in scopes:
    scope_body = {
        "class": "NewRelationshipRequestBody",
        "properties": {"class": "AssignmentScopeProperties", "assignmentType": "Contributor"},
    }
    toms_actor_client.link_assignment_scope(get_guid(community_qualified_name), champion_guid, scope_body)
    print("Added the role to the Sustainability Community")

if responsibility_qualified_name not in related_qualified_names(champion, "governedBy"):
    governed_by_body = {"class": "NewRelationshipRequestBody", "properties": {"class": "GovernedByProperties"}}
    toms_explorer_client.add_gov_definition_to_element(get_guid(responsibility_qualified_name), champion_guid, governed_by_body)
    print("Linked the role to " + responsibility_qualified_name)

if sally_qualified_name not in related_qualified_names(champion, "rolePerformers"):
    toms_actor_client.link_person_role_to_profile(champion_guid, get_guid(sally_qualified_name))
    print("Appointed Sally Counter")

----

## Step 3: Adding procurement staff from the acquired sites

Each acquired site has someone responsible for procurement:

* **Ethan Park**, the Procurement Manager at Austin.
* **Carmen Iliescu**, the Procurement Specialist at Bucharest.

Faith appoints them to the member role of the Procurement Team.  They stay members of their site's office team as well.

----

In [ ]:
team_members_role_guid = get_guid("TeamMembers::" + procurement_team_qualified_name)
team_members_role = faiths_actor_client.get_actor_role_by_guid(team_members_role_guid, output_format="JSON")
current_members = related_qualified_names(team_members_role, "rolePerformers")

for person_qualified_name in ["Person::USA::100180", "Person::RO::EKG-066"]:
    person_guid = get_guid(person_qualified_name)
    if person_guid is None:
        print(f"{person_qualified_name} not found - has adding-acquisition-employees.ipynb been run?")
    elif person_qualified_name in current_members:
        print(f"{person_qualified_name} is already a member of the Procurement Team")
    else:
        faiths_actor_client.link_person_role_to_profile(team_members_role_guid, person_guid)
        print(f"{person_qualified_name} added to the Procurement Team")

----

## Reviewing the results

The Procurement business capability now sits alongside the other business areas, with the Procurement Team as its member and Sally's champion role scoped to it.

----

In [ ]:
procurement = toms_explorer_client.get_element_by_guid(procurement_guid)
print(procurement["properties"]["displayName"] + ": " + procurement["properties"]["description"])
print("Member of: " + ", ".join(related_qualified_names(procurement, "memberOfCollections")))
print("Members: " + ", ".join(related_qualified_names(procurement, "collectionMembers")))
print("Assigned actors: " + ", ".join(related_qualified_names(procurement, "assignedActors")))

champion = toms_actor_client.get_actor_role_by_guid(champion_guid, output_format="JSON")
print()
print(champion["properties"]["displayName"])
for relationship_list_name in ["rolePerformers", "assignmentScope", "governedBy"]:
    print(f"  {relationship_list_name}: " + ", ".join(related_qualified_names(champion, relationship_list_name)))

In [ ]:
report = "| Name | Job title | Site | Role |\n|---|---|---|---|\n"
site_names = {"2": "London", "7": "Austin", "12": "Bucharest"}
for role_name, role_label in [("TeamLeader::", "Team leader"), ("TeamMembers::", "Team member")]:
    role = faiths_actor_client.get_actor_role_by_guid(get_guid(role_name + procurement_team_qualified_name), output_format="JSON")
    for performer in role.get("rolePerformers") or []:
        person = faiths_actor_client.get_actor_profile_by_guid(performer["relatedElement"]["elementHeader"]["guid"], output_format="JSON")
        person_properties = person["properties"]
        if not person_properties["qualifiedName"].startswith("Person:Anon:"):
            work_location = person_properties.get("additionalProperties", {}).get("workLocation")
            report += f"| {person_properties['displayName']} | {person_properties.get('jobTitle', '')} | {site_names.get(work_location, work_location)} | {role_label} |\n"

display(Markdown(report))

----

## Next steps

Sally, Ethan and Carmen can now begin consolidating the supplier lists from the three estates into a single, authoritative set of supplier records - the *Supplier Master Data* that the strategic information supply chains depend on.  As the Procurement Sustainability Champion, Sally will also bring sustainability into how suppliers are chosen.

----
License: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/),
Copyright Contributors to the ODPi Egeria project.